# 👥 Notebook 2 — Segmentación de Perfiles Docentes
**Habilidades demostradas:** K-Means · Silhouette · PCA · Visualización avanzada  

---
**Caso de negocio:** Identificar 4 perfiles distintos de docentes según sus
índices LEPI para diseñar planes de formación diferenciados.


In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
from data_generator import generar_docentes, generar_impacto, resumen_institucional
df = generar_docentes(300)
df_impacto = generar_impacto(15)
print(f"✅ Datos cargados — {len(df)} docentes | {len(df_impacto)} instituciones")
df.head()


## 1. Selección del K óptimo

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

features_km = ['ILD','IRPD','IIP']
scaler = StandardScaler()
X_sc   = scaler.fit_transform(df[features_km])

resultados_k = []
for k in range(2, 9):
    km    = KMeans(n_clusters=k, random_state=42, n_init=15)
    labs  = km.fit_predict(X_sc)
    sil   = silhouette_score(X_sc, labs)
    db    = davies_bouldin_score(X_sc, labs)
    iner  = km.inertia_
    resultados_k.append({'K':k,'Silhouette':sil,'DB_Index':db,'Inercia':iner})

df_k = pd.DataFrame(resultados_k)
print("Métricas por K:")
print(df_k.round(4).to_string(index=False))
print(f"\nK óptimo por Silhouette: K={df_k.loc[df_k['Silhouette'].idxmax(),'K']}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(df_k['K'], df_k['Inercia'], 'bo-', lw=2)
axes[0].set_title('Método del Codo'); axes[0].set_xlabel('K')

axes[1].plot(df_k['K'], df_k['Silhouette'], 'go-', lw=2)
axes[1].set_title('Silhouette Score'); axes[1].set_xlabel('K')

axes[2].plot(df_k['K'], df_k['DB_Index'], 'ro-', lw=2)
axes[2].set_title('Davies-Bouldin (menor=mejor)'); axes[2].set_xlabel('K')
plt.suptitle('Selección del K Óptimo — LEPI', fontweight='bold')
plt.tight_layout(); plt.show()


## 2. Modelo Final K=4 + Etiquetado

In [ ]:
km_final = KMeans(n_clusters=4, random_state=42, n_init=15)
df       = df.copy()
df['cluster'] = km_final.fit_predict(X_sc)
sil_final = silhouette_score(X_sc, df['cluster'])

centros = pd.DataFrame(scaler.inverse_transform(km_final.cluster_centers_),
                       columns=features_km)
centros['ILEPI_est'] = centros.mean(axis=1)
orden = centros['ILEPI_est'].rank(ascending=False).astype(int)
mapa_lab = {
    orden[orden==1].index[0]: '🏆 Líder LEPI',
    orden[orden==2].index[0]: '🧠 Reflexivo',
    orden[orden==3].index[0]: '💡 Innovador',
    orden[orden==4].index[0]: '📚 En Formación'
}
col_perf = {'🏆 Líder LEPI':'#F1C40F','🧠 Reflexivo':'#3498DB',
            '💡 Innovador':'#E74C3C','📚 En Formación':'#95A5A6'}
df['perfil_cluster'] = df['cluster'].map(mapa_lab)
print(f"Silhouette final K=4: {sil_final:.4f}  {'✅' if sil_final>0.5 else '⚠️'}")
print("\nCaracterísticas por perfil:")
print(df.groupby('perfil_cluster')[['ILD','IRPD','IIP','ILEPI','anos_exp']].mean().round(3))


## 3. Visualización PCA 2D + Perfiles

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sc)
df['pca1'] = X_pca[:,0]; df['pca2'] = X_pca[:,1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for perf, grupo in df.groupby('perfil_cluster'):
    axes[0].scatter(grupo['pca1'], grupo['pca2'],
                    label=perf, color=col_perf[perf], alpha=0.7, s=40)
axes[0].set_title(f'Perfiles LEPI — PCA 2D
Silhouette={sil_final:.3f}')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend()

conteos = df['perfil_cluster'].value_counts()
axes[1].pie(conteos.values, labels=conteos.index,
            colors=[col_perf[k] for k in conteos.index],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Distribución de Perfiles Docentes')

plt.suptitle('Segmentación K-Means — App LEPI', fontweight='bold')
plt.tight_layout(); plt.show()


## 4. Rutas de Formación por Perfil

In [ ]:
rutas = {
    '🏆 Líder LEPI':    'Mentoría a pares + Investigación acción',
    '🧠 Reflexivo':     'Diplomado en Innovación Digital + TIC educativas',
    '💡 Innovador':     'Seminario Praxeológico + Comunidades de práctica',
    '📚 En Formación':  'Formación intensiva en Liderazgo + Praxeología + Innovación',
}
print("Plan de formación diferenciado por perfil LEPI:")
print("─"*65)
for perf, ruta in rutas.items():
    n = (df['perfil_cluster']==perf).sum()
    print(f"  {perf} ({n} docentes)")
    print(f"    → {ruta}")


## ✅ Conclusiones Técnicas

### Habilidades demostradas
- **Selección de hiperparámetros**: Elbow method + Silhouette + Davies-Bouldin
- **Reducción de dimensionalidad**: PCA para visualización 2D
- **Etiquetado semántico**: Asignación de perfiles desde centroides

### Valor para la app LEPI
Los 4 perfiles permiten personalizar automáticamente la ruta de formación
de cada docente al ingresar sus índices en la plataforma.
